In [2]:
import sqlite3
import pandas as pd
import os

In [3]:
# get the current working directory
current_working_directory = os.getcwd()

# print output to the console
print(current_working_directory)

/Users/xinyuxu/Desktop/progettoBioinformatica/Causal-Inference-on-eICU-collaborative-research-database-main/notebooks


In [9]:
df_p = pd.read_csv("../data/raw/patient.csv")

df_t = pd.read_csv("../data/raw/treatment.csv")


df_t['treatmentstring'].value_counts()
# Pazienti unici per trattamento
df_t.groupby('treatmentstring')['patientunitstayid'].nunique().sort_values(ascending=False)

pd.DataFrame({
    'righe_totali': df_t['treatmentstring'].value_counts(),
    'pazienti_unici': df_t.groupby('treatmentstring')['patientunitstayid'].nunique()
}).sort_values('pazienti_unici', ascending=False)

,righe_totali,pazienti_unici
treatmentstring,,
pulmonary|ventilation and oxygenation|mechanical ventilation,1088,348
pulmonary|radiologic procedures / bronchoscopy|chest x-ray,750,243
cardiovascular|intravenous fluid|normal saline administration,541,202
pulmonary|ventilation and oxygenation|oxygen therapy (< 40%)|nasal cannula,443,196
cardiovascular|consultations|Cardiology consultation,400,172
...,...,...
infectious diseases|medications|prophylactic antibacterials|for surgery|vancomycin,3,1
burns/trauma|burns|anesthetics|ketamine,2,1
infectious diseases|medications|pneumocystis therapy|dapsone,2,1


In [11]:


# ============================================================
# CARICAMENTO DATI
# ============================================================
df_p = pd.read_csv("../data/raw/patient.csv")
df_t = pd.read_csv("../data/raw/treatment.csv")

# ============================================================
# TABELLA 1 - Top 20 trattamenti per pazienti unici
# ============================================================
tab1 = (
    pd.DataFrame({
        'righe_totali'  : df_t['treatmentstring'].value_counts(),
        'pazienti_unici': df_t.groupby('treatmentstring')['patientunitstayid'].nunique()
    })
    .sort_values('pazienti_unici', ascending=False)
    .reset_index()
    .head(20)
)
tab1['% pazienti'] = (tab1['pazienti_unici'] / df_t['patientunitstayid'].nunique() * 100).round(2)
tab1.index += 1   # indice parte da 1

print("=" * 70)
print("TABELLA 1 — Top 20 trattamenti per pazienti unici")
print("=" * 70)
print(tab1.to_string())


# ============================================================
# TABELLA 2 - Trattati vs non trattati (ventilazione meccanica)
# ============================================================

# Definizione: trattato = almeno 1 riga di mechanical ventilation
MV_STRING = 'pulmonary|ventilation and oxygenation|mechanical ventilation'

pazienti_mv = (
    df_t[df_t['treatmentstring'] == MV_STRING]['patientunitstayid']
    .unique()
)

tutti_pazienti = df_t['patientunitstayid'].unique()

n_trattati     = len(pazienti_mv)
n_non_trattati = len(tutti_pazienti) - n_trattati
n_totale       = len(tutti_pazienti)

tab2 = pd.DataFrame({
    'Gruppo'          : ['Ventilazione meccanica (A=1)', 'Nessuna ventilazione meccanica (A=0)', 'Totale'],
    'N pazienti'      : [n_trattati, n_non_trattati, n_totale],
    '% sul totale'    : [
        round(n_trattati / n_totale * 100, 2),
        round(n_non_trattati / n_totale * 100, 2),
        100.0
    ]
})

print("\n" + "=" * 70)
print("TABELLA 2 — Pazienti trattati vs non trattati (ventilazione meccanica)")
print("=" * 70)
print(tab2.to_string(index=False))


# ============================================================
# TABELLA 3 - Distribuzione temporale del treatmentoffset
#             (focus: ventilazione meccanica)
# ============================================================

df_mv = df_t[df_t['treatmentstring'] == MV_STRING].copy()

# Converti offset in ore per leggibilità
df_mv['offset_ore'] = df_mv['treatmentoffset'] / 60

# Finestre temporali clinicamente rilevanti
bins   = [-np.inf, 0, 6, 12, 24, 48, np.inf]
labels = ['Pre-ricovero (<0h)', '0-6h', '6-12h', '12-24h', '24-48h', '>48h']
df_mv['finestra'] = pd.cut(df_mv['offset_ore'], bins=bins, labels=labels)

tab3 = (
    df_mv.groupby('finestra', observed=True)['patientunitstayid']
    .nunique()
    .reset_index()
    .rename(columns={'patientunitstayid': 'pazienti_unici'})
)
tab3['%'] = (tab3['pazienti_unici'] / n_trattati * 100).round(2)

# Statistiche descrittive offset
stats = df_mv['offset_ore'].describe(percentiles=[.25, .5, .75, .95]).round(2)

print("\n" + "=" * 70)
print("TABELLA 3 — Distribuzione temporale (ventilazione meccanica)")
print("=" * 70)
print("\n--- Distribuzione per finestra temporale ---")
print(tab3.to_string(index=False))
print("\n--- Statistiche descrittive offset (ore) ---")
print(stats.to_string())


# ============================================================
# TABELLA 4 - Missing values e qualità dei dati
# ============================================================

def missing_summary(df, nome):
    total = len(df)
    summary = pd.DataFrame({
        'colonna'      : df.columns,
        'tipo'         : df.dtypes.values,
        'non_nulli'    : df.count().values,
        'nulli'        : df.isnull().sum().values,
        '% missing'    : (df.isnull().mean() * 100).round(2).values,
        'unici'        : df.nunique().values,
    })
    summary.insert(0, 'dataset', nome)
    return summary

tab4_t = missing_summary(df_t, 'treatment')
tab4_p = missing_summary(df_p, 'patient')
tab4   = pd.concat([tab4_t, tab4_p], ignore_index=True)

print("\n" + "=" * 70)
print("TABELLA 4 — Missing values e qualità dei dati")
print("=" * 70)
print(tab4.to_string(index=False))

# Nota su offset negativi
n_offset_neg = (df_t['treatmentoffset'] < 0).sum()
print(f"\nNota: {n_offset_neg} righe con treatmentoffset < 0 "
      f"({round(n_offset_neg/len(df_t)*100,2)}% del totale) "
      f"→ trattamento documentato prima dell'ingresso in UTI.")

TABELLA 1 — Top 20 trattamenti per pazienti unici
                                                                  treatmentstring  righe_totali  pazienti_unici  % pazienti
1                    pulmonary|ventilation and oxygenation|mechanical ventilation          1088             348       18.22
2                      pulmonary|radiologic procedures / bronchoscopy|chest x-ray           750             243       12.72
3                   cardiovascular|intravenous fluid|normal saline administration           541             202       10.58
4      pulmonary|ventilation and oxygenation|oxygen therapy (< 40%)|nasal cannula           443             196       10.26
5                            cardiovascular|consultations|Cardiology consultation           400             172        9.01
6                  pulmonary|ventilation and oxygenation|non-invasive ventilation           330             168        8.80
7               endocrine|glucose metabolism|insulin|sliding scale administration 